# Multi-Agent Event Planner &mdash; Running Sub-Agents in Parallel

This notebook is the **parallel** companion to `3.multi-agent-eventplanner.ipynb`.

The venue search and the playlist lookup are **independent** sub-tasks &mdash; there is no
reason to wait for one before starting the other. Here we show:

1. The **timing intuition**: the same two sub-agents run sequentially vs. concurrently with `asyncio.gather`.
2. **Async coordinator tools** so the coordinator can overlap the two searches.
3. A **parallel coordinator** that is prompted to emit both tool calls in one turn.
4. The **caveats** &mdash; when parallelism does (and does not) apply.

> Why this works: an agent step is **I/O-bound** (it mostly *waits* on the model endpoint).
> `await ...ainvoke()` releases the event loop while waiting, so two calls can be in flight at once.

In [ ]:
%run ../../langchain_common.py

## Setup &mdash; tools, state, and sub-agents

Identical to the sequential notebook, so this notebook stands on its own.

In [ ]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_KEY"))

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters
      (e.g., use 'capacite' instead of 'capacite').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:
    """Query the database for playlist information"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

In [ ]:
from langchain.agents import AgentState

class EventState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

In [ ]:
# Venue agent (uses web search)
venue_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
    You have a suggested limit of 5 web searches. Count every web_search call you make.
    After 5 searches, you should stop searching and summarize the best options you have found so far.
    """
)

In [ ]:
# Playlist agent (uses the SQLite database)
playlist_agent = create_agent(
    model=llm,
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## 1. The core idea &mdash; sequential vs. parallel timing

This first comparison is **model-independent**: we call the two sub-agents *directly* (not through
the coordinator), once one-after-the-other and once concurrently. This isolates the pure speedup
so the contrast is repeatable and doesn't depend on how the LLM decides to batch its tool calls.

- **Sequential** total time  &asymp;  `venue + playlist`
- **Parallel** total time  &asymp;  `max(venue, playlist)`

In [ ]:
import asyncio, time

VENUE_Q    = "Find wedding venues in Gujranwala for 100 guests"
PLAYLIST_Q = "Find jazz tracks for a wedding playlist with multiple artists"

def run_venue_sync():
    return venue_agent.invoke({"messages": [HumanMessage(content=VENUE_Q)]})

def run_playlist_sync():
    return playlist_agent.invoke({"messages": [HumanMessage(content=PLAYLIST_Q)]})

# --- Sequential: one after the other ---
t0 = time.perf_counter()
v = run_venue_sync()
p = run_playlist_sync()
seq = time.perf_counter() - t0
print(f"Sequential : {seq:6.1f}s   (venue THEN playlist)")

In [ ]:
# --- Parallel: overlap the two waits with asyncio.gather ---
# Top-level await works directly inside a Jupyter cell.
t0 = time.perf_counter()
v, p = await asyncio.gather(
    venue_agent.ainvoke({"messages": [HumanMessage(content=VENUE_Q)]}),
    playlist_agent.ainvoke({"messages": [HumanMessage(content=PLAYLIST_Q)]}),
)
par = time.perf_counter() - t0
print(f"Parallel   : {par:6.1f}s   (venue AND playlist together)")
print(f"Speedup    : {seq/par:6.1f}x")

**What just happened?** In the sequential block, Python blocks on the first `.invoke()` until it
finishes, then starts the second. In the parallel block, `.ainvoke()` returns a coroutine; 
`asyncio.gather` starts *both* and lets their network waits overlap. Because the work is I/O-bound,
the total collapses to roughly the slower of the two &mdash; no threads required.

## 2. Async coordinator tools

To get the same overlap *inside* the coordinator, the tools that call the sub-agents must be
`async` and `await` their sub-agent. Then, when the coordinator emits **both** tool calls in a
single turn, LangGraph's tool executor awaits them concurrently instead of one-by-one.

`update_state` stays synchronous and is deliberately **not** parallelized: the two searches depend
on the values it writes, so it must run alone first.

In [ ]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
async def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = await venue_agent.ainvoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
async def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = await playlist_agent.ainvoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> Command:
    """Update the state when you know all of the values: origin, destination, guest_count, genre.
    This tool must be called alone, without any other tool calls. It must complete and return to make
    the information available to other tools."""
    return Command(update={
        "origin": origin,
        "destination": destination,
        "guest_count": guest_count,
        "genre": genre,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
    )

## 3. The parallel coordinator

Same coordinator as before, with one crucial addition to the system prompt: once the state is set,
call `search_venues` and `suggest_playlist` **together in the same turn** so they run in parallel.
The model must emit both tool calls at once &mdash; that is the gating factor for route (1) parallelism.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory_saver = MemorySaver()

coordinator_parallel = create_agent(
    model=llm,
    tools=[search_venues, suggest_playlist, update_state],
    state_schema=EventState,
    checkpointer=memory_saver,
    system_prompt="""
    You are an event planner & coordinator.
    STEP 1: First gather the information you need, then call update_state ALONE
            (no other tool calls in that turn) to save origin, destination, guest_count, genre.
    STEP 2: After update_state returns, delegate to your specialists. search_venues and
            suggest_playlist are INDEPENDENT of each other, so call BOTH of them together
            in the SAME turn (emit both tool calls at once) so they run in parallel.
            Never wait for one before requesting the other.
    STEP 3: Once you have both answers, coordinate the perfect event for me.
    """
)

In [ ]:
async def send_message_parallel(msg: str, thread_id: str):
    """Invoke the coordinator asynchronously so concurrent tool calls actually overlap."""
    response = await coordinator_parallel.ainvoke(
        {"messages": [HumanMessage(content=msg)]},
        config={
            "configurable": {"thread_id": thread_id},
            "tags": ["EventPlannerParallel"],
            "recursion_limit": 40,
        },
    )
    return response["messages"]

In [ ]:
thread_id = new_conversation_id()

t0 = time.perf_counter()
msg_list = await send_message_parallel(
    "I'm from Lahore and I'd like an event in Gujranwala for 100 guests. "
    "Also, create a playlist with jazz genre. Make sure that playlist has multiple artists.",
    thread_id,
)
print(f"\nTotal coordinator time: {time.perf_counter() - t0:.1f}s\n")
pp(msg_list[-1].content)

## 4. Caveats &mdash; when parallelism applies

- **Sync tools + `.invoke()` can never overlap.** You need `async def` tools *and* `await ...ainvoke()`
  on both the sub-agents and the coordinator.
- **The model must batch the calls.** Parallelism inside the coordinator only happens when the LLM
  returns both tool calls in one AI message. The prompt above nudges it to; if a run still
  serializes, open the **MLflow / LangSmith trace** and look at the two tool spans &mdash; overlapping
  bars = parallel, stacked bars = sequential. Cell 1 always proves the achievable ceiling.
- **Only parallelize independent work.** `update_state` runs alone because the searches depend on it.
  If the playlist depended on the chosen venue, they would have to stay ordered too.
- **You pay for concurrency.** Running both sub-agents at once means paying for both at once and
  it can hit provider rate limits &mdash; parallelism trades money and quota for wall-clock speed.

> For guaranteed structural parallelism that does not rely on the model batching, you can drop below
> `create_agent` and build an explicit `StateGraph` with parallel edges into a merge node, or use the
> `Send` API for dynamic map-reduce over many items.